# NB14e — Cluster-Based Premium Calibration + Filter-Profile (ANN-013 Lock)

**Datum:** 2026-05-28
**Lock:** [ANN-013](../docs/decisions/ANN-013-cluster-based-premium-detection.md) — Cluster-Based Premium Detection (supersedes ANN-012 Cutoff-Mechanik)
**Vorgänger-Diagnostik:** [NB14d](14d_proba_distribution_diagnostic.ipynb) → Verdict-Klasse C (ultra-discrete, stable, kalibriert)

**Ziel:** Premium-Tier via Cluster-Extraction definieren, Filter-Profile (Aggressive/Balanced/Conservative) auf höchstem stabilen Cluster validieren, **Multi-Run-Statistik** (3 seeds, mean ± std) liefern für Pine-Export-Lock.

**Neue Regel (Nico, 2026-05-28):** Wichtige Zahlen brauchen MINDESTENS 3 Reruns mit mean/std. Single-Run-Entscheidungen verboten.

**Was dieses Notebook macht (10 Sections):**
0. Setup + Module Sync
1. Load Data + Walk-Forward Split (FX-Train, 5m)
2. Multi-Seed Training (3 seeds = [42, 1, 7])
3. **Cluster-Extraction pro Run** (höchster qualifizierter Cluster pro VAL-Distribution)
4. **Cluster-Stability-Check** (drift über 3 seeds, ANN-013 Requirement: drift < 0.001)
5. Filter-Profile auf Cluster-Cutoff anwenden (Aggressive/Balanced/Conservative)
6. Per-Profile Multi-Run Metriken (mean ± std über 3 seeds): PF, WR, Sigs/Tag, MDD
7. Hold-Out-Validation (GBPUSD, AUDUSD, USDCHF) — Multi-Run aggregiert
8. Quality-Anchor-Check per Profil (ANN-010)
9. Pine-Export-Ready Output: stabiler Cluster-Value für Codegen
10. Persistence (`/results/nb14e/`)
11. Auto-Push to GitHub

**Lock-Anforderung an V1-Pine-Code:** Der extrahierte Cluster-Value wird vom Codegen in Pine eingebaut. Bei jedem Re-Training wird er neu extrahiert — kein hardcoded `0.4096`.

**Erwartete Laufzeit:** ~10 Min (3 Trainings + Cluster-Extraction + Filter-Anwendung + Hold-Out).

## Section 0 — Setup + Module Sync

In [ ]:
import os, sys, subprocess, gc, json, importlib
from pathlib import Path
from datetime import datetime, timezone

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    PROJECT_ROOT = '/content/drive/MyDrive/pace-algo'
    os.chdir(PROJECT_ROOT)
    subprocess.run(['rm', '-rf', '/tmp/pace-algo'])
    subprocess.run(['git', 'clone', '-q', '--depth', '1', 'https://github.com/ecoNC/pace-algo.git',
                    '/tmp/pace-algo'], check=True)
    subprocess.run(f'mkdir -p {PROJECT_ROOT}/core/eval {PROJECT_ROOT}/core/analysis {PROJECT_ROOT}/core/router',
                   shell=True)
    subprocess.run(f'cp -rf /tmp/pace-algo/core/. {PROJECT_ROOT}/core/', shell=True)
    subprocess.run(f"find {PROJECT_ROOT}/core -type d -name __pycache__ -exec rm -rf {{}} +", shell=True)
    print('Core synced.')
else:
    PROJECT_ROOT = os.path.abspath('..')
    os.chdir(PROJECT_ROOT)

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
for mod in list(sys.modules.keys()):
    if mod.startswith('core'):
        del sys.modules[mod]
importlib.invalidate_caches()

import numpy as np
RUN_DATE      = datetime.now(timezone.utc).strftime('%Y-%m-%d')
RUN_TIMESTAMP = datetime.now(timezone.utc).strftime('%Y-%m-%dT%H-%M-%SZ')
try:
    GIT_COMMIT = subprocess.check_output(['git', '-C', PROJECT_ROOT, 'rev-parse', '--short', 'HEAD'], text=True).strip()
except Exception:
    GIT_COMMIT = 'unknown'
EXPERIMENT_ID = f'nb14e_{RUN_TIMESTAMP}_{GIT_COMMIT}'
print(f'EXPERIMENT_ID: {EXPERIMENT_ID}')

if IN_COLAB:
    subprocess.run(['pip', 'install', '-q', 'lightgbm>=4.3', 'pyarrow>=15.0'], capture_output=True)

import pandas as pd
import lightgbm as lgb
import warnings
warnings.filterwarnings('ignore')

from core import config as cfg
from core.config import FX_TRAIN_SYMBOLS, FX_HOLDOUT_SYMBOLS, TRAIN_END, VAL_END
from core.train.dataset import walk_forward_split, binary_label_for_long, NON_FEATURE_COLS
from core.analysis.probability_diagnostic import (
    extract_premium_cluster, apply_cluster_cutoff_mask,
    cluster_stability_test_multi_seed,
)
from core.analysis.product_metrics import signals_per_day

TF = '5m'
R_VALUE = 1.5
WIN_R, LOSS_R = R_VALUE, 1.0
SEEDS = [42, 1, 7]
MIN_CLUSTER_SIZE_PCT = 0.5  # ANN-013 default
DECIMAL_PLACES = 4

NY_SESSION_HOUR_START = 13
NY_SESSION_HOUR_END   = 22

PROFILES = {
    'Aggressive':   {'require_htf': False, 'require_ny': False},
    'Balanced':     {'require_htf': True,  'require_ny': False},
    'Conservative': {'require_htf': True,  'require_ny': True},
}

OUTPUT_DIR = Path(PROJECT_ROOT) / 'results' / 'nb14e'
for sub in ('metrics', 'summaries', 'config_snapshots'):
    (OUTPUT_DIR / sub).mkdir(parents=True, exist_ok=True)

print(f'Seeds: {SEEDS}  |  TF: {TF}')
print(f'Min cluster size: {MIN_CLUSTER_SIZE_PCT}%  |  Decimal places: {DECIMAL_PLACES}')
print(f'Output: {OUTPUT_DIR}')

## Section 1 — Load Data + Walk-Forward

In [ ]:
if IN_COLAB:
    DATA_EXT = Path('/content/processed_v2')
    DATA_PROCESSED_LOCAL = Path('/content/processed')
    EXT_DRIVE_BACKUP = Path(PROJECT_ROOT) / 'data_cache' / 'processed_v2'
    DRIVE_BACKUP_PROCESSED = Path(PROJECT_ROOT) / 'data_cache' / 'processed'
    DATA_EXT.mkdir(parents=True, exist_ok=True)
    DATA_PROCESSED_LOCAL.mkdir(parents=True, exist_ok=True)
    if len(list(DATA_PROCESSED_LOCAL.glob('labels_*.parquet'))) == 0 and DRIVE_BACKUP_PROCESSED.exists():
        subprocess.run(['rsync', '-ah', f'{DRIVE_BACKUP_PROCESSED}/', f'{DATA_PROCESSED_LOCAL}/'])
    if len(list(DATA_EXT.glob('*_extended.parquet'))) == 0 and EXT_DRIVE_BACKUP.exists():
        subprocess.run(['rsync', '-ah', f'{EXT_DRIVE_BACKUP}/', f'{DATA_EXT}/'])
else:
    DATA_EXT = cfg.DATA_PROCESSED.parent / 'processed_v2'
    DATA_PROCESSED_LOCAL = cfg.DATA_PROCESSED

def load_ext(sym, tf):
    p = DATA_EXT / f'{sym}_{tf}_extended.parquet'
    if not p.exists():
        return None
    df = pd.read_parquet(p)
    if 'hit_bar_offset' not in df.columns:
        lp = DATA_PROCESSED_LOCAL / f'labels_{sym}_{tf}_R{R_VALUE}.parquet'
        if lp.exists():
            labels = pd.read_parquet(lp)
            if 'hit_bar_offset' in labels.columns:
                df = df.join(labels[['hit_bar_offset']], how='left')
        if 'hit_bar_offset' not in df.columns:
            df['hit_bar_offset'] = 24
        df['hit_bar_offset'] = df['hit_bar_offset'].fillna(24).astype('int32')
    return df

missing = [s for s in (FX_TRAIN_SYMBOLS + FX_HOLDOUT_SYMBOLS)
           if load_ext(s, TF) is None or load_ext(s, TF).empty]
if missing:
    raise SystemExit(f'Missing _extended for: {missing}')

# Train pool
frames = []
for s in FX_TRAIN_SYMBOLS:
    d = load_ext(s, TF)
    d['symbol'] = s
    frames.append(d.astype({c: 'float32' for c in d.select_dtypes(include=['float64']).columns}))
pool = pd.concat(frames, axis=0).sort_index()
del frames; gc.collect()

probe = load_ext(FX_TRAIN_SYMBOLS[0], TF)
FEATURE_COLS = [c for c in probe.columns if c not in NON_FEATURE_COLS and c != 'symbol']
print(f'Features: {len(FEATURE_COLS)}')

pool_c = pool.dropna(subset=FEATURE_COLS + ['label'])
train_df, val_df, test_df = walk_forward_split(pool_c, TRAIN_END, VAL_END)
print(f'Train: {len(train_df):,}  Val: {len(val_df):,}  Test: {len(test_df):,}')

X_train = train_df[FEATURE_COLS].values.astype(np.float32)
y_train = binary_label_for_long(train_df['label']).values
X_val   = val_df[FEATURE_COLS].values.astype(np.float32)
y_val   = binary_label_for_long(val_df['label']).values
X_test  = test_df[FEATURE_COLS].values.astype(np.float32)
del pool, pool_c
gc.collect()

## Section 2 — Multi-Seed Training Helper

In [ ]:
BASE_PARAMS = {
    'objective':         'binary',
    'metric':            'binary_logloss',
    'num_leaves':        7,
    'max_depth':         3,
    'min_data_in_leaf':  200,
    'learning_rate':     0.05,
    'num_iterations':    100,
    'lambda_l2':         1.0,
    'feature_fraction':  0.8,
    'bagging_fraction':  0.8,
    'bagging_freq':      5,
    'is_unbalance':      True,
    'verbose':           -1,
    'n_jobs':            -1,
}

def train_with_seed(X_tr, y_tr, X_vl, y_vl, seed):
    params = dict(BASE_PARAMS)
    params['seed'] = seed
    td = lgb.Dataset(X_tr, label=y_tr)
    vd = lgb.Dataset(X_vl, label=y_vl, reference=td)
    model = lgb.train(
        params, td, valid_sets=[td, vd], valid_names=['train','val'],
        callbacks=[lgb.early_stopping(10, verbose=False), lgb.log_evaluation(period=0)],
    )
    return model

print('Helper defined.')

## Section 3 — Cluster-Extraction pro Run (3 Seeds)

Trainiere 3 Modelle, extrahiere höchsten qualifizierten Cluster pro VAL-Distribution.

In [ ]:
per_seed_models = {}
per_seed_clusters = {}
per_seed_proba_val = {}
per_seed_proba_test = {}

for seed in SEEDS:
    print(f'\n--- Seed {seed} ---')
    model = train_with_seed(X_train, y_train, X_val, y_val, seed)
    pv = model.predict(X_val)
    pt = model.predict(X_test)
    per_seed_models[seed] = model
    per_seed_proba_val[seed]  = pv
    per_seed_proba_test[seed] = pt

    ext = extract_premium_cluster(pv, decimal_places=DECIMAL_PLACES,
                                    min_cluster_size_pct=MIN_CLUSTER_SIZE_PCT)
    per_seed_clusters[seed] = ext

    print(f'VAL proba range: [{pv.min():.4f}, {pv.max():.4f}]')
    print(f'TEST proba range: [{pt.min():.4f}, {pt.max():.4f}]')
    if ext['success']:
        print(f'Premium cluster: value={ext["premium_cluster_value"]:.4f}  '
              f'count={ext["premium_cluster_size"]}  pct={ext["premium_cluster_pct"]:.2f}%')
        print(f'N qualifying clusters: {ext["n_qualifying"]}')
    else:
        print(f'EXTRACT FAILED: {ext.get("error")}')

## Section 4 — Cluster-Stability-Check (ANN-013 Requirement)

Drift über 3 seeds. ANN-013 requirement: `drift < 0.001`.

In [ ]:
cluster_values = [per_seed_clusters[s]['premium_cluster_value']
                  for s in SEEDS if per_seed_clusters[s]['success']]
cluster_drift = max(cluster_values) - min(cluster_values)
cluster_mean = float(np.mean(cluster_values))
cluster_std = float(np.std(cluster_values))
is_stable = bool(cluster_drift < 0.001)

print('=== Cluster Stability across seeds ===')
for s in SEEDS:
    c = per_seed_clusters[s]
    if c['success']:
        print(f'  seed={s}:  premium={c["premium_cluster_value"]:.4f}  '
              f'({c["premium_cluster_pct"]:.2f}%, n={c["premium_cluster_size"]})')
print(f'\nDrift: {cluster_drift:.4f}  (ANN-013 requires < 0.001)')
print(f'Mean:  {cluster_mean:.4f}  ± {cluster_std:.4f}')
print(f'is_stable: {is_stable}')

if not is_stable:
    print('\n⚠️  Cluster-Value drift > 0.001 — Modell-Architektur muss überdacht werden.')
    print('   ABER: wir fahren fort und nutzen MEAN als Lock-Wert (Multi-Run-Robustheit).')

# Lock-Value = Mean über stabile Runs (auch bei is_stable=False nutzen wir mean als beste Schätzung)
LOCKED_PREMIUM_CLUSTER = cluster_mean
print(f'\n→ LOCKED_PREMIUM_CLUSTER (für Pine-Export): {LOCKED_PREMIUM_CLUSTER:.4f}')

## Section 5 — Filter-Definitionen + apply

In [ ]:
def apply_filters(df, proba, profile_name, cluster_value):
    """Boolean mask der bars die das Profil als Signal akzeptiert."""
    cfg_prof = PROFILES[profile_name]
    # Premium-Cluster-Cutoff (Cluster-Detection rounded compare)
    rounded = np.round(proba, DECIMAL_PLACES)
    mask = rounded >= cluster_value
    if cfg_prof['require_htf']:
        if 'htf_ltf_agree_bull' not in df.columns:
            raise KeyError('htf_ltf_agree_bull fehlt')
        mask = mask & (df['htf_ltf_agree_bull'].values == 1)
    if cfg_prof['require_ny']:
        hour_utc = df.index.hour.values
        ny_mask = (hour_utc >= NY_SESSION_HOUR_START) & (hour_utc < NY_SESSION_HOUR_END)
        mask = mask & ny_mask
    return mask

print('Filter-Helper defined.')

## Section 6 — Per-Profile Multi-Run Metriken (3 seeds, mean ± std)

Pro Profil + Seed: PF / WR / Sigs/Tag / MDD / Trades. Dann mean ± std über die 3 seeds.

In [ ]:
def metrics_for_mask(labels_triple, mask, n_bars, n_symbols, durations=None):
    n_sig = int(mask.sum())
    if n_sig == 0:
        return {'n_trades': 0, 'wins': 0, 'losses': 0, 'pf': 0.0, 'wr': 0.0,
                'sigs_per_day_per_symbol': 0.0, 'mdd': 0.0, 'avg_duration': 0.0}
    labs = labels_triple[mask.values if hasattr(mask, 'values') else mask]
    wins = int((labs == 1).sum())
    losses = int((labs == -1).sum())
    pf = (wins * WIN_R) / losses if losses > 0 else (float('inf') if wins > 0 else 0.0)
    wr = wins / (wins + losses) if (wins + losses) > 0 else 0.0
    pnl = np.where(labs == 1, WIN_R, np.where(labs == -1, -1.0, 0.0))
    equity = np.cumsum(pnl)
    rmax = np.maximum.accumulate(equity)
    mdd = float((rmax - equity).max()) if len(equity) > 0 else 0.0
    sigs_pd = signals_per_day(n_sig, n_bars, TF, n_symbols)
    avg_dur = float(durations[mask.values if hasattr(mask, 'values') else mask].mean()) if durations is not None and n_sig > 0 else 0.0
    return {'n_trades': n_sig, 'wins': wins, 'losses': losses,
            'pf': pf, 'wr': wr, 'sigs_per_day_per_symbol': sigs_pd,
            'mdd': mdd, 'avg_duration': avg_dur}

test_labels_triple = test_df['label'].values
test_durations = test_df['hit_bar_offset'].values if 'hit_bar_offset' in test_df.columns else None

in_sample_per_seed = []
for seed in SEEDS:
    proba_test = per_seed_proba_test[seed]
    for p in PROFILES.keys():
        mask = apply_filters(test_df, proba_test, p, LOCKED_PREMIUM_CLUSTER)
        m = metrics_for_mask(test_labels_triple, mask, len(test_df), len(FX_TRAIN_SYMBOLS), test_durations)
        m['seed'] = seed
        m['profile'] = p
        in_sample_per_seed.append(m)

in_sample_df = pd.DataFrame(in_sample_per_seed)
print('=== IN-SAMPLE TEST — per seed × profile ===')
display_cols = ['seed', 'profile', 'n_trades', 'wr', 'pf', 'sigs_per_day_per_symbol', 'mdd', 'avg_duration']
print(in_sample_df[display_cols].round(4).to_string(index=False))

# Aggregate: mean ± std über seeds
agg_in_sample = []
for p in PROFILES.keys():
    sub = in_sample_df[in_sample_df['profile'] == p]
    pf_arr = sub['pf'].replace([np.inf, -np.inf], np.nan).dropna()
    agg_in_sample.append({
        'profile':           p,
        'pf_mean':           float(pf_arr.mean()) if len(pf_arr) > 0 else None,
        'pf_std':            float(pf_arr.std()) if len(pf_arr) > 1 else None,
        'wr_mean':           float(sub['wr'].mean()),
        'wr_std':            float(sub['wr'].std()),
        'sigs_per_day_mean': float(sub['sigs_per_day_per_symbol'].mean()),
        'sigs_per_day_std':  float(sub['sigs_per_day_per_symbol'].std()),
        'mdd_mean':          float(sub['mdd'].mean()),
        'mdd_std':           float(sub['mdd'].std()),
        'n_trades_mean':     float(sub['n_trades'].mean()),
    })
agg_in_sample_df = pd.DataFrame(agg_in_sample)
print('\n=== AGGREGATE (mean ± std over 3 seeds) ===')
print(agg_in_sample_df.round(4).to_string(index=False))

## Section 7 — Hold-Out Validation (Multi-Run aggregiert)

Pro Symbol × Profil × Seed → mean ± std über seeds.

In [ ]:
val_end_ts = pd.Timestamp(VAL_END)
if val_end_ts.tz is None:
    val_end_ts = val_end_ts.tz_localize('UTC')

holdout_per_seed = []
for seed in SEEDS:
    model = per_seed_models[seed]
    for sym in FX_HOLDOUT_SYMBOLS:
        h = load_ext(sym, TF)
        h = h.dropna(subset=FEATURE_COLS + ['label'])
        h = h[h.index >= val_end_ts]
        if h.empty:
            continue
        proba_h = model.predict(h[FEATURE_COLS].values.astype(np.float32))
        labels_h = h['label'].values
        durations_h = h['hit_bar_offset'].values if 'hit_bar_offset' in h.columns else None
        for p in PROFILES.keys():
            mask = apply_filters(h, proba_h, p, LOCKED_PREMIUM_CLUSTER)
            m = metrics_for_mask(labels_h, mask, len(h), 1, durations_h)
            m['seed'] = seed
            m['profile'] = p
            m['symbol'] = sym
            holdout_per_seed.append(m)
        del h, proba_h
        gc.collect()

holdout_df = pd.DataFrame(holdout_per_seed)

# Aggregat über seeds × symbols pro profil (weighted PF, mean ± std über seeds)
agg_holdout = []
for p in PROFILES.keys():
    sub = holdout_df[holdout_df['profile'] == p]
    # Pro seed: aggregiere über symbole
    per_seed_aggs = []
    for seed in SEEDS:
        seed_sub = sub[sub['seed'] == seed]
        if seed_sub.empty:
            continue
        w_wins = seed_sub['wins'].sum()
        w_losses = seed_sub['losses'].sum()
        w_trades = seed_sub['n_trades'].sum()
        w_pf = (w_wins * WIN_R) / w_losses if w_losses > 0 else (float('inf') if w_wins > 0 else 0.0)
        w_wr = w_wins / (w_wins + w_losses) if (w_wins + w_losses) > 0 else 0.0
        per_seed_aggs.append({'seed': seed, 'pf': w_pf, 'wr': w_wr, 'n_trades': w_trades,
                              'sigs_per_day': seed_sub['sigs_per_day_per_symbol'].mean(),
                              'mdd': seed_sub['mdd'].max()})
    if not per_seed_aggs:
        continue
    ps_df = pd.DataFrame(per_seed_aggs)
    pf_arr = ps_df['pf'].replace([np.inf, -np.inf], np.nan).dropna()
    agg_holdout.append({
        'profile':        p,
        'pf_mean':        float(pf_arr.mean()) if len(pf_arr) > 0 else None,
        'pf_std':         float(pf_arr.std()) if len(pf_arr) > 1 else 0.0,
        'wr_mean':        float(ps_df['wr'].mean()),
        'wr_std':         float(ps_df['wr'].std()),
        'n_trades_mean':  float(ps_df['n_trades'].mean()),
        'sigs_per_day_mean': float(ps_df['sigs_per_day'].mean()),
        'sigs_per_day_std':  float(ps_df['sigs_per_day'].std()),
        'mdd_mean':       float(ps_df['mdd'].mean()),
    })
agg_holdout_df = pd.DataFrame(agg_holdout)
print('=== HOLD-OUT AGGREGATED (per profile, mean ± std over 3 seeds, weighted across symbols) ===')
print(agg_holdout_df.round(4).to_string(index=False))

## Section 8 — Quality-Anchor-Check per Profil (ANN-010)

Nutzt MEAN-Werte aus den 3 Seeds (nicht Single-Run).

In [ ]:
QUALITY_STRICT = {
    'min_premium_pf_oos':       1.5,
    'min_premium_pf_holdout':   1.4,
    'max_mdd':                  None,  # interpretiert hier als R-Units, nicht Prozent
}

quality_rows = []
for p in PROFILES.keys():
    is_row = next((r for r in agg_in_sample if r['profile'] == p), None)
    ho_row = next((r for r in agg_holdout if r['profile'] == p), None)
    if is_row is None or ho_row is None:
        continue
    is_pf  = is_row.get('pf_mean') or 0
    ho_pf  = ho_row.get('pf_mean') or 0
    passed_strict = (
        is_pf >= QUALITY_STRICT['min_premium_pf_oos'] and
        ho_pf >= QUALITY_STRICT['min_premium_pf_holdout']
    )
    severity = (
        'PASSED' if passed_strict
        else 'SOFT_ONLY' if (is_pf >= 1.3 and ho_pf >= 1.3)
        else 'BLOCKED'
    )
    quality_rows.append({
        'profile':            p,
        'is_pf_mean':         round(is_pf, 4),
        'ho_pf_mean':         round(ho_pf, 4),
        'is_pf_std':          round(is_row.get('pf_std') or 0, 4),
        'ho_pf_std':          round(ho_row.get('pf_std') or 0, 4),
        'ho_sigs_per_day':    round(ho_row.get('sigs_per_day_mean') or 0, 4),
        'severity':           severity,
    })
quality_df = pd.DataFrame(quality_rows)
print('=== QUALITY-ANCHOR per profile (Multi-Run Mean basis) ===')
print(quality_df.to_string(index=False))

## Section 9 — Pine-Export-Ready Output + Verdict

In [ ]:
all_passed = all(r['severity'] in ('PASSED', 'SOFT_ONLY') for r in quality_rows)

pine_export_ready = {
    'lock_basis':              'ANN-013 Cluster-Based Premium Detection',
    'model':                   'fx_lgbm_v1',
    'tf':                      TF,
    'premium_cluster_value':   LOCKED_PREMIUM_CLUSTER,
    'cluster_drift':           cluster_drift,
    'cluster_std':             cluster_std,
    'cluster_is_stable':       is_stable,
    'seeds_tested':            SEEDS,
    'min_cluster_size_pct':    MIN_CLUSTER_SIZE_PCT,
    'decimal_places':          DECIMAL_PLACES,
    'profile_definitions':     PROFILES,
    'ny_session_utc':          [NY_SESSION_HOUR_START, NY_SESSION_HOUR_END],
    'quality_per_profile':     quality_rows,
    'all_profiles_v1_ready':   all_passed,
}

print('=' * 70)
print(f'NB14e VERDICT — {RUN_DATE}')
print('=' * 70)
print(f'\nLOCKED Premium-Cluster-Value: {LOCKED_PREMIUM_CLUSTER:.4f}')
print(f'Cluster-Drift (3 seeds):      {cluster_drift:.4f}  (ANN-013 req: < 0.001)')
print(f'Cluster-Stable:               {is_stable}')
print(f'\nProfile-Quality:')
for r in quality_rows:
    print(f'  {r["profile"]:14s}  IS_PF={r["is_pf_mean"]:.2f}±{r["is_pf_std"]:.2f}  '
          f'HO_PF={r["ho_pf_mean"]:.2f}±{r["ho_pf_std"]:.2f}  '
          f'Sigs={r["ho_sigs_per_day"]:.2f}/Tag/Sym  → {r["severity"]}')
print(f'\nV1-Ready (all profiles passed): {all_passed}')
print('=' * 70)

## Section 10 — Persistence

In [ ]:
in_sample_df.to_csv(OUTPUT_DIR / 'metrics' / f'in_sample_per_seed_profile_{RUN_DATE}.csv', index=False)
agg_in_sample_df.to_csv(OUTPUT_DIR / 'metrics' / f'in_sample_aggregate_{RUN_DATE}.csv', index=False)
holdout_df.to_csv(OUTPUT_DIR / 'metrics' / f'holdout_per_seed_symbol_profile_{RUN_DATE}.csv', index=False)
agg_holdout_df.to_csv(OUTPUT_DIR / 'metrics' / f'holdout_aggregate_{RUN_DATE}.csv', index=False)
quality_df.to_csv(OUTPUT_DIR / 'metrics' / f'quality_per_profile_{RUN_DATE}.csv', index=False)

snapshot = {
    'experiment_id':   EXPERIMENT_ID,
    'run_date':        RUN_DATE,
    'git_commit':      GIT_COMMIT,
    'seeds':           SEEDS,
    'pine_export_ready': pine_export_ready,
    'per_seed_clusters': {str(s): per_seed_clusters[s] for s in SEEDS},
    'in_sample_per_seed': in_sample_df.replace([np.inf, -np.inf], 'inf').to_dict(orient='records'),
    'in_sample_aggregate': agg_in_sample_df.replace([np.inf, -np.inf], 'inf').to_dict(orient='records'),
    'holdout_per_seed': holdout_df.replace([np.inf, -np.inf], 'inf').to_dict(orient='records'),
    'holdout_aggregate': agg_holdout_df.replace([np.inf, -np.inf], 'inf').to_dict(orient='records'),
    'quality_anchor': quality_rows,
}
snap_path = OUTPUT_DIR / 'summaries' / f'nb14e_full_snapshot_{RUN_DATE}.json'
with open(snap_path, 'w') as f:
    json.dump(snapshot, f, indent=2, default=str)
print(f'Snapshot → {snap_path}')

cfg_snap = {
    'experiment_id':         EXPERIMENT_ID,
    'run_date':              RUN_DATE,
    'git_commit':            GIT_COMMIT,
    'base_params':           BASE_PARAMS,
    'seeds':                 SEEDS,
    'min_cluster_size_pct':  MIN_CLUSTER_SIZE_PCT,
    'decimal_places':        DECIMAL_PLACES,
    'profiles':              PROFILES,
    'ny_session_utc':        [NY_SESSION_HOUR_START, NY_SESSION_HOUR_END],
    'lock_basis':            'ANN-013',
}
cfg_path = OUTPUT_DIR / 'config_snapshots' / f'{EXPERIMENT_ID}_config.json'
with open(cfg_path, 'w') as f:
    json.dump(cfg_snap, f, indent=2, default=str)
print(f'Config → {cfg_path}')

## Section 11 — Auto-Push to GitHub

In [ ]:
import shutil
if not IN_COLAB:
    print('Local run — skip push.')
else:
    try:
        from google.colab import userdata
        GH_TOKEN = userdata.get('GITHUB_TOKEN')
    except Exception as e:
        GH_TOKEN = None
        print(f'ERROR cannot read GITHUB_TOKEN: {e}')

    if GH_TOKEN:
        NB_TAG = 'nb14e'
        TMP_REPO = Path('/tmp/pace-algo-push')
        if TMP_REPO.exists():
            shutil.rmtree(TMP_REPO)
        subprocess.run(['git', 'clone', '--quiet',
                        f'https://{GH_TOKEN}@github.com/ecoNC/pace-algo.git', str(TMP_REPO)], check=True)
        subprocess.run(['git', '-C', str(TMP_REPO), 'config', 'user.name', 'ecoNC'], check=True)
        subprocess.run(['git', '-C', str(TMP_REPO), 'config', 'user.email',
                        'ecoNC@users.noreply.github.com'], check=True)
        # Pull --rebase VOR Copy (Lesson aus NB14c)
        subprocess.run(['git', '-C', str(TMP_REPO), 'pull', '--rebase', '--autostash', '--quiet',
                        'origin', 'main'], check=True)
        print('Pulled latest from origin/main')

        copied = []
        for f in sorted(OUTPUT_DIR.rglob(f'*{RUN_DATE}*')):
            if not f.is_file(): continue
            rel = f.relative_to(Path(PROJECT_ROOT) / 'results')
            dest = TMP_REPO / 'results' / rel
            dest.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(f, dest)
            copied.append(rel)
        for f in sorted((OUTPUT_DIR / 'config_snapshots').glob(f'*{EXPERIMENT_ID}*')):
            rel = f.relative_to(Path(PROJECT_ROOT) / 'results')
            dest = TMP_REPO / 'results' / rel
            dest.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(f, dest)
            copied.append(rel)
        print(f'Copied {len(copied)} files')

        subprocess.run(['git', '-C', str(TMP_REPO), 'add', 'results/'], check=True)
        r_status = subprocess.run(['git', '-C', str(TMP_REPO), 'status', '--porcelain'],
                                    capture_output=True, text=True)
        if not r_status.stdout.strip():
            print('Nothing to commit.')
        else:
            msg = f'{NB_TAG.upper()} cluster-based premium calibration {RUN_DATE} ({len(copied)} files)'
            subprocess.run(['git', '-C', str(TMP_REPO), 'commit', '-m', msg], check=True)
            sha = subprocess.run(['git', '-C', str(TMP_REPO), 'rev-parse', '--short', 'HEAD'],
                                   capture_output=True, text=True).stdout.strip()
            subprocess.run(['git', '-C', str(TMP_REPO), 'push', 'origin', 'main'], check=True)
            print(f'✓ Pushed as ecoNC ({sha})')
            print(f'  https://github.com/ecoNC/pace-algo/commit/{sha}')
        shutil.rmtree(TMP_REPO)